# 🚀 Raasta-3D: 100% Free Video to 3D Gaussian Splatting Pipeline
### Namma Space Competition • Techfest, IIT Bombay & Juspay

This Google Colab notebook automates the entire end-to-end processing pipeline on a **free GPU**:
1. **Video Ingestion:** Smartphone monocular video upload.
2. **COLMAP Structure-from-Motion:** Automated camera pose estimation and keypoint matching via `ns-process-data`.
3. **3D Gaussian Splatting Training:** Trains real-time radiance fields using `splatfacto` in ~10-15 minutes.
4. **Export & Optimization:** Exports `.ply` / `.splat` point cloud ready for **SuperSplat** and **Raasta-3D Web Viewer**.

### Step 1: Verify Free GPU Runtime
Ensure you have enabled a GPU in Colab: **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.

In [ ]:
!nvidia-smi

### Step 2: Install Dependencies & Nerfstudio
Installs PyTorch, CUDA toolkit dependencies, Ninja compiler, and Nerfstudio.

In [ ]:
# Upgrade pip and install core libraries
!pip install --upgrade pip
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install ninja

# Install Nerfstudio (splatfacto engine)
!pip install nerfstudio

### Step 3: Upload Indoor Video Capture
Upload your video captured using the **Raasta-3D Smartphone SOP** (2-4 minutes, 4K or 1080p, loop closure).
Name your file `indoor_capture.mp4` or upload via the Colab files panel.

In [ ]:
from google.colab import files
import os

print("Please upload your indoor capture video (mp4/mov):")
uploaded = files.upload()

# Rename uploaded video to standard input name
for fn in uploaded.keys():
  ext = os.path.splitext(fn)[1]
  os.rename(fn, f"indoor_capture{ext}")
  input_video_path = f"indoor_capture{ext}"
  print(f"Saved capture video to {input_video_path}")
  break

### Step 4: Video Processing & COLMAP Structure-from-Motion
Nerfstudio's `ns-process-data` automatically:
- Cuts video into non-blurry keyframe photos
- Runs COLMAP to compute camera poses, focal lengths, and sparse point cloud

In [ ]:
!ns-process-data video \
  --data indoor_capture.mp4 \
  --output-dir /content/processed_data \
  --num-frames-target 300 \
  --matching-method sequential \
  --sfm-tool colmap

### Step 5: Train 3D Gaussian Splatting (splatfacto)
Trains Gaussian ellipsoids using differentiable splatting. 15,000 iterations typically takes 10-15 minutes on a T4 GPU.

In [ ]:
!ns-train splatfacto \
  --data /content/processed_data \
  --max-num-iterations 15000 \
  --output-dir /content/outputs

### Step 6: Export to 3D Gaussian Splat (.ply)
Find the trained config and export into standard `.ply` format.

In [ ]:
import glob

config_files = glob.glob("/content/outputs/**/config.yml", recursive=True)
if config_files:
  latest_config = config_files[-1]
  print(f"Found training config: {latest_config}")
  !ns-export gaussian-splat --load-config {latest_config} --output-dir /content/exported_splat
  print("Gaussian splat exported successfully!")
else:
  print("No config file found. Please verify training completed.")

### Step 7: Download Model for Raasta-3D
Download the generated `.ply` file to your local computer.

In [ ]:
ply_files = glob.glob("/content/exported_splat/**/*.ply", recursive=True)
if ply_files:
  print(f"Downloading {ply_files[0]}...")
  files.download(ply_files[0])
else:
  print("Exported ply not found.")

### Step 8: Next Steps (Cleaning & Loading)
1. Open the free web tool: **[SuperSplat](https://playcanvas.com/supersplat/editor)**.
2. Drag the `.ply` into SuperSplat -> use the Box Selection tool to delete floaters or background outside walls.
3. Export as `.splat` or `.ply`.
4. Place in `Raasta-3D/public/models/namma-space.ply` OR directly drag and drop into the **Raasta-3D Web Viewer** using the **Load Model** button!